# Tutorial 16 — Inference: KV Cache, Flash Attention & Speculative Decoding

**Series:** Training Language Models from Scratch: A Hacker's Guide  
**Part V — Inference**  
**Follows:** Tutorial 15 (Quantization)  
**Precedes:** Tutorial 17 (Capstone)

---

## What This Tutorial Covers

Training a model is a one-time cost. Inference is paid on every request,
forever. A model that generates 10 tokens/sec is not deployable. A model
that generates 500 tokens/sec can serve real users. The difference is
almost entirely algorithmic — the same weights, the same hardware, orders
of magnitude apart in throughput.

This tutorial covers the three techniques that make the difference:

1. **KV Cache** — the single most important inference optimization. Why
   autoregressive generation is quadratic in sequence length without it,
   and how caching the key/value projections reduces it to linear.
   Implementing a stateful cache, the memory/speed tradeoff, cache
   eviction strategies.

2. **Flash Attention** — rewriting the attention computation to be
   IO-aware. Why naive attention is memory-bandwidth-bound, not
   compute-bound. Tiling, the online softmax trick, why Flash Attention
   is both faster AND more memory-efficient than standard attention.

3. **Speculative Decoding** — using a small draft model to propose tokens
   that the large model verifies in parallel. Why verification is cheap,
   why acceptance rate matters, the exact acceptance criterion that
   preserves the target model's distribution exactly.

At the end: a `FastInferenceEngine` that combines all three into a single
coherent inference stack, and a throughput benchmark that measures tokens/sec
at each optimization stage.

---

## 1. The Autoregressive Generation Problem

Language model generation is sequential: to generate token $t+1$, you
need token $t$. The forward pass for token $t+1$ runs on the full sequence
$[x_1, \ldots, x_t, y_{t+1\text{ prompt}}]$ — even though tokens
$x_1, \ldots, x_t$ have not changed since the previous step.

In the attention computation, each query at position $t$ attends to all
previous keys and values:

$$\text{Attn}(Q_t, K_{1:t}, V_{1:t}) = \text{softmax}\!\left(\frac{Q_t K_{1:t}^\top}{\sqrt{d_k}}\right) V_{1:t}$$

Without caching, computing $K_{1:t}$ and $V_{1:t}$ requires running the
key and value projection matrices over all $t$ previous tokens —
$O(t)$ work per generated token, $O(t^2)$ total for a sequence of length $t$.

For a 2048-token sequence, this means the 2048th token takes 2048× more
compute than the first. [[Throughput collapses as sequences get longer.]{.mark}

---

## 2. The KV Cache

The fix: after computing $K_i$ and $V_i$ for position $i$, store them.
When generating position $t+1$, only compute $Q_{t+1}$, $K_{t+1}$,
$V_{t+1}$ for the new token, then look up $K_{1:t}$ and $V_{1:t}$ from
the cache.

```
Without KV cache:                With KV cache:
  step t: compute K_{1:t}          step t: compute K_t, V_t only
           compute V_{1:t}                   load K_{1:t-1}, V_{1:t-1}
           compute Q_{1:t}                   compute Q_t
           run full attention                run attention with cached K/V
  Cost: O(t) per step               Cost: O(1) per step (new token)
  Total: O(t^2)                     Total: O(t) + O(t) cache memory
```

[The tradeoff: memory grows linearly with sequence length.]{.mark} For a model with
$L$ layers, $H$ heads, head dimension $d_h$, and sequence length $T$:

$$\text{KV cache size} = 2 \times L \times H \times d_h \times T \times \text{bytes\_per\_value}$$

[[For our nano model ($L=6$, $H=6$, $d_h=64$, BF16=2 bytes):]{.underline}
$$2 \times 6 \times 6 \times 64 \times T \times 2 = 9216 \times T \text{ bytes}$$

At $T=1024$: ~9MB. At $T=4096$: ~36MB. Manageable.

For a 7B model ($L=32$, $H=32$, $d_h=128$, BF16=2 bytes):
$$2 \times 32 \times 32 \times 128 \times T \times 2 = 524288 \times T \text{ bytes}$$

At $T=4096$: ~2GB just for the KV cache. This is why KV cache management
is a real constraint in production serving.

### Implementing the KV cache

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass, field
from typing import Optional


@dataclass
class KVCache:
    """
    Stores key and value tensors for all layers.

    k_cache[layer]: (B, H, T_cached, d_h)
    v_cache[layer]: (B, H, T_cached, d_h)

    On each generation step, new K/V are appended along the T dimension.
    """
    n_layers:    int
    k_cache:     list[Optional[torch.Tensor]] = field(default_factory=list)
    v_cache:     list[Optional[torch.Tensor]] = field(default_factory=list)
    seq_len:     int = 0   # current number of cached tokens

    def __post_init__(self):
        self.k_cache = [None] * self.n_layers
        self.v_cache = [None] * self.n_layers

    def update(
        self,
        layer_idx: int,
        new_k:     torch.Tensor,   # (B, H, T_new, d_h)
        new_v:     torch.Tensor,   # (B, H, T_new, d_h)
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Append new K/V to the cache for this layer.
        Returns the full cached K/V tensors.
        """
        if self.k_cache[layer_idx] is None:
            self.k_cache[layer_idx] = new_k
            self.v_cache[layer_idx] = new_v
        else:
            self.k_cache[layer_idx] = torch.cat(
                [self.k_cache[layer_idx], new_k], dim=2
            )
            self.v_cache[layer_idx] = torch.cat(
                [self.v_cache[layer_idx], new_v], dim=2
            )

        if layer_idx == 0:
            self.seq_len = self.k_cache[0].size(2)

        return self.k_cache[layer_idx], self.v_cache[layer_idx]

    def clear(self):
        """Reset cache between sequences."""
        self.k_cache = [None] * self.n_layers
        self.v_cache = [None] * self.n_layers
        self.seq_len = 0

    def memory_bytes(self) -> int:
        """Current cache memory usage in bytes."""
        total = 0
        for k, v in zip(self.k_cache, self.v_cache):
            if k is not None:
                total += k.numel() * k.element_size()
                total += v.numel() * v.element_size()
        return total

    def evict_oldest(self, n_tokens: int):
        """
        Remove the oldest n_tokens from the cache.
        Used when cache exceeds max_seq_len.
        """
        for i in range(self.n_layers):
            if self.k_cache[i] is not None:
                self.k_cache[i] = self.k_cache[i][:, :, n_tokens:, :]
                self.v_cache[i] = self.v_cache[i][:, :, n_tokens:, :]
        self.seq_len -= n_tokens

### Modifying attention to use the cache

The attention module needs a code path that:
1. Computes K and V only for the new token(s)
2. Reads the full K/V from the cache
3. Computes attention between the new query and all cached K/V

In [ ]:
def attention_with_cache(
    q:          torch.Tensor,    # (B, H, T_new, d_h)
    k_new:      torch.Tensor,    # (B, H, T_new, d_h)
    v_new:      torch.Tensor,    # (B, H, T_new, d_h)
    cache:      KVCache,
    layer_idx:  int,
    is_causal:  bool = True,
) -> torch.Tensor:
    """
    Attention computation using the KV cache.
    For generation (T_new=1), this reduces to a single vector-matrix product.
    """
    # Append new K/V and retrieve full cache
    k_full, v_full = cache.update(layer_idx, k_new, v_new)
    # k_full: (B, H, T_cached, d_h)
    # q:      (B, H, T_new,    d_h)

    d_h    = q.size(-1)
    scale  = d_h ** -0.5

    # Attention scores: (B, H, T_new, T_cached)
    scores = torch.matmul(q, k_full.transpose(-2, -1)) * scale

    # Causal mask — only needed during prefill (T_new > 1)
    # During generation (T_new=1), the query can attend to all cached tokens
    if is_causal and q.size(2) > 1:
        T_new    = q.size(2)
        T_cached = k_full.size(2)
        # New tokens can only attend to positions up to their own position
        mask = torch.ones(T_new, T_cached, device=q.device, dtype=torch.bool)
        mask = torch.triu(mask, diagonal=T_cached - T_new + 1)
        scores = scores.masked_fill(mask.unsqueeze(0).unsqueeze(0), -1e9)

    weights = F.softmax(scores, dim=-1)
    return torch.matmul(weights, v_full)   # (B, H, T_new, d_h)


def generate_with_cache(
    model,
    tokenizer,
    prompt:         str,
    max_new_tokens: int   = 200,
    temperature:    float = 0.8,
    top_k:          int   = 50,
    device:         torch.device = None,
) -> str:
    """
    Autoregressive generation with KV cache.

    Two phases:
    1. Prefill: run the full prompt through the model in one forward pass,
       populating the KV cache for all prompt tokens.
    2. Decode: generate one token at a time, using the cache.
    """
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    cache = KVCache(n_layers=model.config.n_layers)

    # ---- Phase 1: Prefill ----
    prompt_ids = torch.tensor(
        [tokenizer.encode(prompt)], dtype=torch.long, device=device
    )
    T_prompt = prompt_ids.size(1)

    with torch.no_grad():
        # Full forward pass — populates cache for all prompt positions
        logits = model.forward_with_cache(prompt_ids, cache, start_pos=0)
        # logits: (B, T_prompt, V) — we only need the last position
        next_logits = logits[:, -1, :]   # (B, V)

    # ---- Phase 2: Decode ----
    generated = []
    for _ in range(max_new_tokens):
        # Sample next token
        if temperature > 0:
            probs  = F.softmax(next_logits / temperature, dim=-1)
            if top_k > 0:
                # Zero out all but top-k probabilities
                top_vals, _ = probs.topk(top_k, dim=-1)
                probs[probs < top_vals[:, -1:]] = 0
                probs = probs / probs.sum(dim=-1, keepdim=True)
            next_token = torch.multinomial(probs, num_samples=1)
        else:
            next_token = next_logits.argmax(dim=-1, keepdim=True)

        tok_id = next_token.item()
        if tok_id == tokenizer.eos_id:
            break

        generated.append(tok_id)

        # Single-token forward pass with cache
        with torch.no_grad():
            next_logits = model.forward_with_cache(
                next_token, cache,
                start_pos=T_prompt + len(generated) - 1
            )[:, -1, :]   # (B, V)

    cache.clear()
    return tokenizer.decode(generated)

### Cache eviction strategies

When the generated sequence exceeds `max_seq_len`, the cache must be
truncated. Three common strategies:

In [ ]:
class CacheEvictionStrategy:
    """Base class for cache eviction strategies."""

    def should_evict(self, cache: KVCache, max_len: int) -> bool:
        return cache.seq_len > max_len

    def evict(self, cache: KVCache, max_len: int):
        raise NotImplementedError


class SlidingWindowEviction(CacheEvictionStrategy):
    """
    Keep only the most recent max_len tokens.
    Simple and effective for most chat use cases.
    The model loses context beyond max_len tokens.
    """
    def evict(self, cache, max_len):
        overflow = cache.seq_len - max_len
        cache.evict_oldest(overflow)


class SinkTokenEviction(CacheEvictionStrategy):
    """
    StreamingLLM (Xiao et al. 2023) strategy:
    Always keep the first `n_sink` tokens (attention sinks) plus the
    most recent max_len - n_sink tokens. Preserves initial context
    which anchors the model's attention patterns.
    """

    def __init__(self, n_sink: int = 4):
        self.n_sink = n_sink

    def evict(self, cache, max_len):
        # Keep first n_sink tokens and last (max_len - n_sink) tokens
        # Remove tokens in the middle
        n_keep_recent = max_len - self.n_sink
        n_drop        = cache.seq_len - max_len

        for i in range(cache.n_layers):
            if cache.k_cache[i] is None:
                continue
            k = cache.k_cache[i]   # (B, H, T, d_h)
            v = cache.v_cache[i]

            sink_k    = k[:, :, :self.n_sink, :]
            sink_v    = v[:, :, :self.n_sink, :]
            recent_k  = k[:, :, self.n_sink + n_drop:, :]
            recent_v  = v[:, :, self.n_sink + n_drop:, :]

            cache.k_cache[i] = torch.cat([sink_k, recent_k], dim=2)
            cache.v_cache[i] = torch.cat([sink_v, recent_v], dim=2)

        cache.seq_len -= n_drop

---

## 3. Flash Attention

Standard attention is IO-bound, not compute-bound. Let's understand why.

### The memory bottleneck in standard attention

For a sequence of length $T$ and head dimension $d_h$, standard attention:

1. Computes $S = QK^\top / \sqrt{d_h}$ — shape $(T, T)$
2. Stores $S$ to HBM (GPU main memory)
3. Reads $S$ from HBM to compute softmax
4. Stores softmax weights $P$ to HBM
5. Reads $P$ from HBM to compute $PV$

Steps 2–4 each read/write an $O(T^2)$ matrix to HBM. For $T=2048$:
that is $2048^2 \times 2 \approx 8\text{MB}$ per read/write, per layer,
per forward pass. With 6 layers and bidirectional passes: ~300MB of HBM
traffic just for attention intermediates.

HBM bandwidth on an A100 is 2 TB/s. SRAM bandwidth is ~20 TB/s.
[The bottleneck is the HBM round-trips, not the FLOPs.]{.underline}

### Flash Attention: tiling to stay in SRAM

Flash Attention (Dao et al., 2022) rewrites the attention computation so
that intermediate matrices never need to be written to HBM. It processes
the attention in tiles that fit in SRAM.

The key mathematical ingredient: the **online softmax**[^online_softmax]

[^online_softmax]: The online softmax is a numerically stable one-pass algorithm for computing $\text{softmax}(x) \cdot V$ without ever materializing the full $T$-length attention weight vector. It maintains a running maximum $m$, running normalizer $l$, and running output $o$, correcting each as a new element arrives. This is the mathematical foundation of Flash Attention tiling. — computing
softmax in a single pass without materializing the full attention matrix.

**Online softmax:** For a vector $x \in \mathbb{R}^T$, standard softmax
requires two passes: one to find the maximum (for numerical stability)
and one to compute the exponentials. The online algorithm processes one
element at a time, maintaining a running maximum $m$ and a running sum $l$:

```
m_0 = -∞, l_0 = 0, o_0 = 0
For i = 1 to T:
    m_i = max(m_{i-1}, x_i)
    l_i = l_{i-1} * exp(m_{i-1} - m_i) + exp(x_i - m_i)
    o_i = o_{i-1} * exp(m_{i-1} - m_i) / l_i * l_{i-1} + exp(x_i - m_i) / l_i * v_i
```

At each step, the running output $o_i$ is corrected when a new maximum
is found. After the full pass, $o_T = \text{softmax}(x) \cdot V$ — the
attention output — without ever storing the full $T$-length attention weights.

In [ ]:
def flash_attention_reference(
    Q:  torch.Tensor,   # (B, H, T, d_h)
    K:  torch.Tensor,   # (B, H, T, d_h)
    V:  torch.Tensor,   # (B, H, T, d_h)
    block_size: int = 64,
    causal:     bool = True,
) -> torch.Tensor:
    """
    Reference implementation of Flash Attention in pure PyTorch.
    This is NOT as fast as the real CUDA implementation — it is here
    to show the algorithm. The real version uses custom CUDA kernels.

    For production use: torch.nn.functional.scaled_dot_product_attention()
    which calls the optimized Flash Attention implementation automatically.
    """
    B, H, T, d_h = Q.shape
    scale         = d_h ** -0.5
    O             = torch.zeros_like(Q)   # output accumulator
    L             = torch.zeros(B, H, T, device=Q.device)   # log-sum-exp

    # Tile over query blocks
    for q_start in range(0, T, block_size):
        q_end = min(q_start + block_size, T)
        Q_blk = Q[:, :, q_start:q_end, :]   # (B, H, Bq, d_h)
        Bq    = q_end - q_start

        m_i   = torch.full((B, H, Bq), float('-inf'), device=Q.device)
        l_i   = torch.zeros(B, H, Bq, device=Q.device)
        O_i   = torch.zeros(B, H, Bq, d_h, device=Q.device)

        # Tile over key/value blocks
        for kv_start in range(0, T, block_size):
            kv_end = min(kv_start + block_size, T)
            K_blk  = K[:, :, kv_start:kv_end, :]  # (B, H, Bkv, d_h)
            V_blk  = V[:, :, kv_start:kv_end, :]

            # Causal mask: skip future KV blocks entirely
            if causal and kv_start > q_end:
                break

            # Compute attention scores for this tile
            S_blk = torch.matmul(Q_blk, K_blk.transpose(-2, -1)) * scale
            # (B, H, Bq, Bkv)

            # Apply causal mask within the tile
            if causal:
                q_idx  = torch.arange(q_start, q_end, device=Q.device)
                kv_idx = torch.arange(kv_start, kv_end, device=Q.device)
                mask   = q_idx.unsqueeze(1) < kv_idx.unsqueeze(0)
                S_blk  = S_blk.masked_fill(mask.unsqueeze(0).unsqueeze(0), -1e9)

            # Online softmax update
            m_new  = torch.maximum(m_i, S_blk.max(dim=-1).values)
            exp_S  = torch.exp(S_blk - m_new.unsqueeze(-1))
            l_new  = torch.exp(m_i - m_new) * l_i + exp_S.sum(dim=-1)

            # Rescale previous output and add new contribution
            O_i    = (torch.exp(m_i - m_new).unsqueeze(-1) * O_i
                      + torch.matmul(exp_S, V_blk))

            m_i, l_i = m_new, l_new

        # Normalize and write output block
        O[:, :, q_start:q_end, :] = O_i / l_i.unsqueeze(-1)
        L[:, :, q_start:q_end]    = m_i + torch.log(l_i)

    return O


def scaled_dot_product_attention_fast(Q, K, V, is_causal=True):
    """
    Use PyTorch's built-in Flash Attention — automatically selects
    the fastest available implementation (Flash Attention 2 on supported
    hardware, otherwise falls back to math attention).

    This is what you should actually use in your model.
    """
    return F.scaled_dot_product_attention(Q, K, V, is_causal=is_causal)

### Replacing standard attention with Flash Attention

The change to the model is minimal — replace the explicit softmax attention
computation with `F.scaled_dot_product_attention`:

In [ ]:
# In tutorial_02.py MultiHeadAttention.forward():

# BEFORE (standard attention):
scores   = torch.matmul(Q, K.transpose(-2, -1)) * scale
scores   = scores.masked_fill(causal_mask, -1e9)
weights  = F.softmax(scores, dim=-1)
output   = torch.matmul(weights, V)

# AFTER (Flash Attention):
output = F.scaled_dot_product_attention(
    Q, K, V,
    attn_mask=None,
    dropout_p=0.0,
    is_causal=True,
)
# That's it. One line. Same result, ~3× faster, ~O(T) memory instead of O(T^2).

**Memory complexity comparison:**

| Method | Attention memory | Notes |
|---|---|---|
| Standard | $O(T^2)$ | Stores full attention matrix |
| Flash Attention | $O(T)$ | Never materializes full matrix |
| With KV cache | $O(T)$ | Cached K/V per layer |

For $T=8192$: standard attention stores $8192^2 \times 6 \times 2 \approx 800\text{MB}$
of attention intermediates. [Flash Attention stores essentially nothing.]{.mark}

---

## 4. Speculative Decoding

KV cache and Flash Attention fix the per-token cost. Speculative decoding
addresses a different problem: autoregressive generation is sequential —
you cannot generate token $t+1$ until you have token $t$. On a GPU with
thousands of cores, generating one token at a time wastes nearly all of
the available parallelism.

**The idea:** use a small, fast **draft model** to propose $k$ tokens ahead.
Then verify all $k$ proposals with the large **target model** in a single
parallel forward pass. If the target model agrees with the draft, all $k$
tokens are accepted at once. If it disagrees at position $j$, accept tokens
up to $j-1$ and resample from the target at position $j$.

[The key insight: verification is cheap because all $k$ tokens are verified
in a single batched forward pass.]{.mark} Sampling from the large model $k$ times
would require $k$ sequential forward passes. Verification requires one.

### The acceptance criterion

Naïve rejection: accept token $t$ if the draft model's choice matches the
target model's greedy choice. This changes the distribution — you get a
mix of draft and target model behavior, not the pure target distribution.

**The correct acceptance criterion** (Leviathan et al., 2023) preserves
the exact target distribution:

Accept draft token $\tilde{x}$ at position $t$ with probability:

$$\alpha_t = \min\!\left(1,\; \frac{p_{\text{target}}(\tilde{x} \mid x_{<t})}{p_{\text{draft}}(\tilde{x} \mid x_{<t})}\right)$$

If rejected, sample a correction token from:

$$p_{\text{corrected}}(x) \propto \max\!\left(0,\; p_{\text{target}}(x) - p_{\text{draft}}(x)\right)$$

[This guarantees that the output distribution is exactly $p_{\text{target}}$,
regardless of draft model quality.]{.underline} A worse draft model just has lower
acceptance rate; it never corrupts the distribution.

In [ ]:
def speculative_decode_step(
    target_logits: torch.Tensor,   # (1, k+1, V) — target model on draft tokens
    draft_logits:  torch.Tensor,   # (1, k, V)   — draft model when sampling
    draft_tokens:  torch.Tensor,   # (1, k)       — proposed tokens
    temperature:   float = 1.0,
) -> tuple[torch.Tensor, int]:
    """
    One step of speculative decoding: verify k draft tokens against the target.

    Returns:
        accepted_tokens: (1, n_accepted) — accepted token IDs (n_accepted <= k+1)
        n_accepted:      how many draft tokens were accepted
    """
    k = draft_tokens.size(1)
    V = target_logits.size(-1)

    # Compute probability distributions
    target_probs = F.softmax(target_logits[:, :k, :] / max(temperature, 1e-6),
                              dim=-1)   # (1, k, V)
    draft_probs  = F.softmax(draft_logits / max(temperature, 1e-6),
                              dim=-1)   # (1, k, V)

    accepted       = []
    n_accepted     = 0

    for i in range(k):
        token    = draft_tokens[0, i].item()
        p_target = target_probs[0, i, token].item()
        p_draft  = draft_probs[0, i, token].item()

        # Accept with probability min(1, p_target / p_draft)
        acceptance_prob = min(1.0, p_target / (p_draft + 1e-10))

        if torch.rand(1).item() < acceptance_prob:
            accepted.append(token)
            n_accepted += 1
        else:
            # Rejected: sample correction token from residual distribution
            residual = torch.clamp(
                target_probs[0, i] - draft_probs[0, i], min=0
            )
            residual_sum = residual.sum()
            if residual_sum > 1e-8:
                residual = residual / residual_sum
                correction = torch.multinomial(residual, num_samples=1).item()
            else:
                # Fallback: sample from target
                correction = torch.multinomial(
                    target_probs[0, i], num_samples=1
                ).item()
            accepted.append(correction)
            break

    # If all k tokens were accepted, also sample the (k+1)-th token
    # from the target model's prediction at position k
    if n_accepted == k:
        bonus_probs = F.softmax(
            target_logits[0, k, :] / max(temperature, 1e-6), dim=-1
        )
        bonus = torch.multinomial(bonus_probs, num_samples=1).item()
        accepted.append(bonus)

    accepted_tensor = torch.tensor(
        [accepted], dtype=torch.long, device=draft_tokens.device
    )
    return accepted_tensor, n_accepted


def speculative_generate(
    target_model,
    draft_model,
    tokenizer,
    prompt:          str,
    max_new_tokens:  int   = 200,
    k:               int   = 4,      # draft tokens per step
    temperature:     float = 0.8,
    device:          torch.device = None,
) -> tuple[str, dict]:
    """
    Full speculative decoding generation loop.

    Returns:
        text:    generated response
        stats:   dict with acceptance_rate, speedup estimate, etc.
    """
    if device is None:
        device = next(target_model.parameters()).device

    target_model.eval()
    draft_model.eval()

    target_cache = KVCache(n_layers=target_model.config.n_layers)
    draft_cache  = KVCache(n_layers=draft_model.config.n_layers)

    prompt_ids = torch.tensor(
        [tokenizer.encode(prompt)], dtype=torch.long, device=device
    )

    # Prefill both models
    with torch.no_grad():
        target_logits = target_model.forward_with_cache(
            prompt_ids, target_cache, start_pos=0
        )
        draft_logits_prefill = draft_model.forward_with_cache(
            prompt_ids, draft_cache, start_pos=0
        )

    generated       = []
    n_total_drafted = 0
    n_total_accepted = 0
    current_pos     = prompt_ids.size(1)

    while len(generated) < max_new_tokens:
        # ---- Draft k tokens ----
        draft_tokens     = []
        draft_logits_all = []

        last_token = torch.tensor(
            [[generated[-1] if generated else prompt_ids[0, -1].item()]],
            dtype=torch.long, device=device
        )
        cur = last_token

        with torch.no_grad():
            for _ in range(k):
                d_logits = draft_model.forward_with_cache(
                    cur, draft_cache, start_pos=current_pos + len(draft_tokens)
                )[:, -1, :]   # (1, V)
                draft_logits_all.append(d_logits)

                probs  = F.softmax(d_logits / max(temperature, 1e-6), dim=-1)
                next_t = torch.multinomial(probs, num_samples=1)
                draft_tokens.append(next_t.item())
                cur = next_t

        draft_tok_tensor  = torch.tensor([draft_tokens], dtype=torch.long,
                                          device=device)
        draft_logits_cat  = torch.stack(draft_logits_all, dim=1)  # (1, k, V)

        # ---- Verify with target model (single forward pass for all k tokens) ----
        with torch.no_grad():
            # Run target on the k draft tokens + 1 (to get prediction after last)
            verify_ids    = torch.cat([last_token, draft_tok_tensor], dim=1)
            target_logits = target_model.forward_with_cache(
                verify_ids, target_cache,
                start_pos=current_pos
            )   # (1, k+1, V)

        # ---- Accept/reject ----
        accepted_tokens, n_acc = speculative_decode_step(
            target_logits, draft_logits_cat, draft_tok_tensor, temperature
        )

        n_total_drafted  += k
        n_total_accepted += n_acc

        # Roll back draft cache to the accepted position
        n_to_evict = k - n_acc
        if n_to_evict > 0:
            draft_cache.evict_oldest(n_to_evict)

        # Commit accepted tokens
        for tok in accepted_tokens[0].tolist():
            if tok == tokenizer.eos_id:
                break
            generated.append(tok)
            current_pos += 1

        if tokenizer.eos_id in accepted_tokens[0].tolist():
            break
        if len(generated) >= max_new_tokens:
            break

    acceptance_rate = n_total_accepted / max(n_total_drafted, 1)
    # Theoretical speedup: mean tokens per target forward pass
    # Without speculative: 1 token per pass
    # With speculative:    acceptance_rate * k + 1 tokens per pass
    mean_tokens_per_pass = acceptance_rate * k + 1
    speedup_estimate     = mean_tokens_per_pass  # relative to no speculation

    stats = {
        'acceptance_rate':      acceptance_rate,
        'mean_tokens_per_pass': mean_tokens_per_pass,
        'speedup_estimate':     speedup_estimate,
        'total_tokens':         len(generated),
        'total_drafted':        n_total_drafted,
        'total_accepted':       n_total_accepted,
    }

    target_cache.clear()
    draft_cache.clear()
    return tokenizer.decode(generated), stats

### Speedup analysis

The theoretical speedup of speculative decoding depends on:
- $k$: number of draft tokens per step
- $\alpha$: mean acceptance rate per token

[[Mean tokens accepted per target forward pass:]{.mark} $\bar{k} = \sum_{i=1}^{k} \alpha^i + \alpha^k$

For $\alpha = 0.8$ and $k = 4$:
$\bar{k} = 0.8 + 0.64 + 0.512 + 0.41 + 0.41 \approx 2.77$

So you get ~2.77 tokens per target forward pass instead of 1 — a 2.77×
speedup, assuming the draft model runs fast enough that its overhead is small.

The draft model must be fast. The practical constraint:

$$\text{speedup} = \frac{\bar{k}}{1 + \text{draft cost fraction}}$$

If the draft model costs 10% of the target model per token and $k=4$:

$$\text{speedup} \approx \frac{2.77}{1 + 0.1 \times 4} = \frac{2.77}{1.4} \approx 1.98$$

For this series, the [nano model (10.7M) can serve as its own draft model]{.underline}
in a smaller configuration — use a model with $L=2$ layers and $d_{\text{model}}=128$
as the draft for the full $L=6$, $d_{\text{model}}=384$ target.

---

## 5. The `FastInferenceEngine`

Combining all three optimizations into a single coherent interface:

In [ ]:
import time

class FastInferenceEngine:
    """
    Combines KV cache, Flash Attention, and speculative decoding
    into a single inference interface.

    Usage:
        engine = FastInferenceEngine(
            model=trained_model,
            tokenizer=tok,
            device=device,
            use_speculative=True,
            draft_model=small_model,
            k=4,
        )
        response = engine.generate("What is photosynthesis?")
    """

    def __init__(
        self,
        model,
        tokenizer,
        device:           torch.device,
        use_speculative:  bool  = False,
        draft_model       = None,
        k:                int   = 4,
        max_cache_len:    int   = 2048,
        eviction_strategy = None,
    ):
        self.model       = model
        self.tokenizer   = tokenizer
        self.device      = device
        self.use_spec    = use_speculative and (draft_model is not None)
        self.draft_model = draft_model
        self.k           = k
        self.max_cache   = max_cache_len
        self.eviction    = eviction_strategy or SlidingWindowEviction()

        # Ensure Flash Attention is used (torch >= 2.0)
        if hasattr(F, 'scaled_dot_product_attention'):
            print("✓ Flash Attention available via F.scaled_dot_product_attention")
        else:
            print("⚠ Flash Attention not available — upgrade to PyTorch >= 2.0")

    def generate(
        self,
        prompt:         str,
        max_new_tokens: int   = 200,
        temperature:    float = 0.8,
        top_k:          int   = 50,
    ) -> tuple[str, dict]:
        """
        Generate a response, returning (text, performance_stats).
        """
        t0 = time.perf_counter()

        if self.use_spec:
            text, spec_stats = speculative_generate(
                self.model, self.draft_model,
                self.tokenizer, prompt,
                max_new_tokens=max_new_tokens,
                k=self.k,
                temperature=temperature,
                device=self.device,
            )
        else:
            text = generate_with_cache(
                self.model, self.tokenizer, prompt,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_k=top_k,
                device=self.device,
            )
            spec_stats = {}

        elapsed    = time.perf_counter() - t0
        n_tokens   = len(self.tokenizer.encode(text))
        tokens_per_sec = n_tokens / max(elapsed, 1e-6)

        stats = {
            'tokens_generated': n_tokens,
            'time_s':           elapsed,
            'tokens_per_sec':   tokens_per_sec,
            **spec_stats,
        }
        return text, stats

    def benchmark(
        self,
        prompts:        list[str],
        max_new_tokens: int = 100,
        temperature:    float = 0.8,
    ) -> dict:
        """
        Run benchmark over a list of prompts and report aggregate stats.
        """
        all_tps = []
        for prompt in prompts:
            _, stats = self.generate(prompt, max_new_tokens=max_new_tokens,
                                      temperature=temperature)
            all_tps.append(stats['tokens_per_sec'])

        return {
            'mean_tps':   float(np.mean(all_tps)),
            'median_tps': float(np.median(all_tps)),
            'p10_tps':    float(np.percentile(all_tps, 10)),
            'p90_tps':    float(np.percentile(all_tps, 90)),
            'n_prompts':  len(prompts),
        }

---

## 6. Throughput Benchmark

Measure tokens/sec at each optimization stage to understand the contribution
of each technique:

In [ ]:
def run_inference_benchmark(
    model_path: str,
    tokenizer,
    device:     torch.device,
    prompts:    list[str] = None,
):
    """
    Benchmark throughput at each optimization stage.
    Prints a table showing the cumulative speedup from each technique.
    """
    from tutorial_02 import GPT, NanoGPTConfig
    import time

    if prompts is None:
        prompts = [
            "Explain the water cycle in detail.",
            "Write a short story about a robot.",
            "What are the main causes of the French Revolution?",
            "Describe how a computer processor works.",
        ] * 5   # 20 prompts total

    config = NanoGPTConfig()
    model  = GPT(config).to(device)
    ckpt   = torch.load(model_path, map_location=device)
    model.load_state_dict(ckpt['model'] if 'model' in ckpt else ckpt)
    model.eval()

    MAX_NEW = 100
    results = {}

    # ---- Baseline: no cache, no flash attention ----
    def baseline_generate(prompt):
        ids  = torch.tensor([tokenizer.encode(prompt)],
                             dtype=torch.long, device=device)
        t0   = time.perf_counter()
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=MAX_NEW, temperature=0.8,
                                  eos_token_id=tokenizer.eos_id)
        return len(out[0]) - len(ids[0]), time.perf_counter() - t0

    tps_baseline = []
    for p in prompts[:10]:
        n, t = baseline_generate(p)
        tps_baseline.append(n / max(t, 1e-6))
    results['Baseline (no cache)'] = float(np.mean(tps_baseline))

    # ---- With KV cache ----
    engine_cache = FastInferenceEngine(model, tokenizer, device,
                                        use_speculative=False)
    bm = engine_cache.benchmark(prompts[:10], max_new_tokens=MAX_NEW)
    results['+ KV Cache'] = bm['mean_tps']

    # ---- With KV cache + Flash Attention ----
    # Flash Attention is automatically used if model uses
    # F.scaled_dot_product_attention — check and report
    if hasattr(F, 'scaled_dot_product_attention'):
        results['+ Flash Attention'] = bm['mean_tps'] * 1.0
        # (In practice the speedup is measured by profiling — here we
        # note it is enabled and the user should profile with torch.profiler)

    # ---- Print results ----
    print(f"\nInference Throughput Benchmark")
    print(f"{'─'*50}")
    print(f"  {'Configuration':30s}  {'Tokens/sec':>12s}  {'Speedup':>8s}")
    print(f"{'─'*50}")
    baseline = results.get('Baseline (no cache)', 1.0)
    for label, tps in results.items():
        speedup = tps / baseline
        print(f"  {label:30s}  {tps:12.1f}  {speedup:7.2f}×")
    print(f"{'─'*50}")

---

## Summary

| Concept | Key detail |
|---|---|
| Without KV cache | $O(t)$ work per token → $O(t^2)$ total. Throughput collapses for long sequences. |
| KV cache | Cache $K_i$, $V_i$ after computing them. $O(1)$ work per new token. |
| KV cache memory | $2LHd_hT \times \text{bytes}$. Linear in sequence length. |
| Prefill vs decode | Prefill: full prompt in one pass, populates cache. Decode: one token at a time. |
| Sliding window eviction | Keep most recent $T_{\max}$ tokens. Simple, loses distant context. |
| Sink token eviction | Keep first $n_\text{sink}$ + most recent. Preserves attention anchors. |
| Flash Attention bottleneck | Standard attention is IO-bound: $O(T^2)$ HBM reads/writes per layer. |
| Online softmax | Compute attention output in one pass without storing the full attention matrix. |
| Flash Attention memory | $O(T)$ vs $O(T^2)$. One line in PyTorch: `F.scaled_dot_product_attention`. |
| Speculative decoding | Draft $k$ tokens with small model, verify in one target forward pass. |
| Acceptance criterion | $\min(1, p_\text{target} / p_\text{draft})$. Preserves exact target distribution. |
| Correction sampling | Sample from $\max(0, p_\text{target} - p_\text{draft})$ on rejection. |
| Speedup formula | Mean tokens per pass = $\sum_{i=1}^k \alpha^i + \alpha^k$. For $\alpha=0.8, k=4$: ~2.77×. |
| Draft model cost | Speculative speedup falls if draft model costs > ~10–15% of target per token. |

---

## Exercises

**1.** Implement a `cache_memory_profiler` that runs generation for sequences
of lengths {64, 128, 256, 512, 1024} and records peak GPU memory at each
length with and without the KV cache. Plot memory vs sequence length for
both cases. Confirm that without the cache, memory grows quadratically
(due to attention intermediates), and with the cache, it grows linearly.

**2.** Verify that `speculative_decode_step` preserves the target distribution.
Run 10000 speculative decoding steps with a draft model that always
proposes the wrong token (acceptance rate = 0). Collect the correction
tokens sampled from the residual distribution. Verify that their empirical
distribution matches $p_\text{target}$ on a small vocabulary test case.

**3.** Implement the `SinkTokenEviction` strategy and compare it to
`SlidingWindowEviction` on a long-context task: generate 2000 tokens
starting from a prompt that establishes a specific fact in the first
sentence. Measure how often each strategy's output references the
initial fact correctly at token 1500. Sink eviction should score higher.

**4.** Profile standard attention vs `F.scaled_dot_product_attention` using
`torch.profiler`. For sequence lengths {128, 256, 512, 1024, 2048},
record wall time and peak memory for both implementations. Plot the
crossover point where Flash Attention's memory savings become significant.

**5.** Build a `draft_model_ablation`: train draft models of sizes
{1-layer, 2-layer, 4-layer} (same hidden dim as the target). Run
speculative decoding with each as the draft and measure acceptance rate
and wall-clock tokens/sec. Find the sweet spot between draft model
quality (acceptance rate) and draft model speed (per-token cost).

**6.** Combine all three optimizations: run the `FastInferenceEngine` with
KV cache + Flash Attention + speculative decoding and report the end-to-end
tokens/sec vs the baseline. Then apply INT8 quantization (Tutorial 15)
to both the target and draft models and re-benchmark. Report the combined
speedup from quantization + KV cache + Flash Attention + speculative
decoding over the unoptimized float32 baseline.